# Does Proxy-Based Data Selection Survive Contact With Reality?

**Reproducible analysis notebook.** Every table and figure in the paper is produced
by running this notebook top to bottom against `results/runs.jsonl`.

Nothing here trains anything. Training ran on a Kaggle Tesla T4 and is recorded in
`notebooks/kaggle_fasttrack.py`; this notebook reads only what that produced.

In [1]:
import glob
import json
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

runs = [json.loads(l) for l in open(ROOT / "results/runs.jsonl") if l.strip()]
ok = [r for r in runs if r.get("status") == "ok"]
print(f"{len(ok)} completed runs, {len(runs) - len(ok)} failed")
print("learning rates:", sorted({r["config"]["learning_rate"] for r in ok}))

23 completed runs, 0 failed
learning rates: [0.0002]


## 1. The frozen split

Created once and committed. The prompt template is hashed into it, and `data.py`
refuses to load if the template later changes -- silent template drift would
invalidate every run without raising anything.

In [2]:
split = json.loads((ROOT / "results/split.json").read_text())
print(f"pool            : {split['pool']}  ({split['pool_size']} examples)")
print(f"train / held-out: {len(split['train_idx'])} / {len(split['held_out_idx'])}")
print(f"split seed      : {split['split_seed']}")
print(f"template hash   : {split['template_hash']}")

pool            : databricks/databricks-dolly-15k  (15011 examples)
train / held-out: 14000 / 1000
split seed      : 0
template hash   : cb11391ee245


## 2. Measured selection cost

The paper's central measurement, and a direct one: no model, no fit. Every method
must compute something over all 14,000 candidates before a single training step can
run.

Wall-clock is comparable only *within* a device. `results/cpu_reference/` records the
same perplexity selection on a laptop CPU (11,943 s) against a T4 (493.7 s) -- 24x
apart, with identical analytical FLOPs.

In [3]:
rows = []
for f in sorted(glob.glob(str(ROOT / "results/selections/*__r0.05__s0.json"))):
    d = json.load(open(f))
    cfg = d["selector_config"]
    scorer = cfg.get("scorer") or cfg.get("embedder") or cfg.get("proxy") or "-"
    rows.append((d["method"], d["cost_class"], d["cost"]["wall_clock_s"],
                 d["cost"]["flops"], scorer))

print(f"{'method':>22} {'cost class':>15} {'wall-clock':>13} {'FLOPs':>11}  scorer")
for m, c, w, fl, sc in sorted(rows, key=lambda r: r[3]):
    print(f"{m:>22} {c:>15} {w:>11.1f} s {fl:>11.3e}  {sc.split('/')[-1]}")

                method      cost class    wall-clock       FLOPs  scorer
                random            free         0.0 s   0.000e+00  -
             diversity   training-free        58.0 s   9.379e+13  all-MiniLM-L6-v2
            perplexity   training-free       493.7 s   7.119e+14  SmolLM-135M
                   ifd   training-free       959.3 s   1.005e+15  SmolLM-135M
   learning_percentage  training-based      1624.7 s   3.570e+15  SmolLM-135M


## 3. Table 1 -- the main grid

Five methods x two ratios x two seeds on Qwen2.5-0.5B. `sel %` is the share of
**total** FLOPs spent selecting rather than training.

In [4]:
grid = [r for r in ok if r["study"] == "fast_grid"]
agg = defaultdict(lambda: {"loss": [], "arc": [], "share": [], "total": []})
for r in grid:
    c, m = r["config"], r["metrics"]
    a = agg[(c["selection_method"], c["ratio"])]
    a["loss"].append(m["held_out_loss"])
    a["total"].append(r["cost"]["total_flops"])
    a["share"].append(r["cost"]["selection_share_of_total_flops"])
    if "arc_easy.acc" in m:
        a["arc"].append(m["arc_easy.acc"])

base = {rat: np.mean(agg[("random", rat)]["loss"]) for rat in (0.05, 0.10)
        if ("random", rat) in agg}

print(f"{'method':>22} {'ratio':>6} {'loss':>8} {'sd':>7} {'d rand':>8} "
      f"{'ARC-E':>7} {'sel %':>7} {'FLOPs':>10}")
for k in sorted(agg, key=lambda k: (k[1], np.mean(agg[k]["loss"]))):
    a = agg[k]
    v = np.array(a["loss"])
    arc = np.mean(a["arc"]) if a["arc"] else float("nan")
    delta = v.mean() - base[k[1]] if k[1] in base else float("nan")
    print(f"{k[0]:>22} {k[1]:>6.2f} {v.mean():>8.4f} {v.std(ddof=1):>7.4f} "
          f"{delta:>+8.4f} {arc:>7.3f} {np.mean(a['share']) * 100:>6.1f}% "
          f"{np.mean(a['total']):>10.2e}")

                method  ratio     loss      sd   d rand   ARC-E   sel %      FLOPs
                   ifd   0.05   1.7801  0.0031  -0.0164   0.622   58.9%   1.71e+15
                random   0.05   1.7965  0.0134  +0.0000   0.632    0.0%   6.63e+14
             diversity   0.05   1.7996  0.0001  +0.0032   0.618   11.0%   8.54e+14
   learning_percentage   0.05   1.8093  0.0056  +0.0129   0.618   83.6%   4.27e+15
            perplexity   0.05   1.9093  0.0197  +0.1128   0.628   72.0%   9.88e+14
                   ifd   0.10   1.7977  0.0075  -0.0048   0.617   42.0%   2.39e+15
                random   0.10   1.8025  0.0044  +0.0000   0.615    0.0%   1.33e+15
             diversity   0.10   1.8103  0.0030  +0.0077   0.618    5.8%   1.63e+15
   learning_percentage   0.10   1.8111  0.0007  +0.0086   0.613   72.2%   4.94e+15
            perplexity   0.10   1.8594  0.0060  +0.0568   0.617   49.4%   1.44e+15


## 4. RQ2 -- the Pareto frontier moves when selection is priced in

Scored on **training compute alone** (how prior work accounts) versus **total
compute**. A method that leaves the frontier was only ever on it because the cost of
choosing its data went unrecorded.

In [5]:
from stats import pareto_frontier

pts = [{"label": f"{r['config']['selection_method']}@{r['config']['ratio']:g}",
        "train": r["cost"]["training_flops"],
        "total": r["cost"]["total_flops"],
        "loss": r["metrics"]["held_out_loss"]} for r in grid]
loss = np.array([p["loss"] for p in pts])

train_only = {pts[i]["label"] for i in
              pareto_frontier(np.array([p["train"] for p in pts]), loss)}
with_sel = {pts[i]["label"] for i in
            pareto_frontier(np.array([p["total"] for p in pts]), loss)}

print("frontier, TRAINING cost only :", sorted(train_only))
print("frontier, TOTAL cost         :", sorted(with_sel))
print()
print("leaves the frontier once selection is priced in:",
      sorted(train_only - with_sel) or "none")

frontier, TRAINING cost only : ['ifd@0.05', 'perplexity@0.05', 'random@0.05']
frontier, TOTAL cost         : ['ifd@0.05', 'random@0.05']

leaves the frontier once selection is priced in: ['perplexity@0.05']


## 5. Is more selected data better?

The scaling law behind the proposed rule assumes held-out loss falls monotonically
with the selection ratio. Checked, rather than assumed.

In [6]:
bym = defaultdict(dict)
for k, a in agg.items():
    bym[k[0]][k[1]] = float(np.mean(a["loss"]))

worse = 0
print(f"{'method':>22} {'5%':>9} {'10%':>9} {'delta':>9}")
for m, d in sorted(bym.items()):
    if 0.05 in d and 0.10 in d:
        delta = d[0.10] - d[0.05]
        worse += delta > 0
        flag = "  WORSE with more data" if delta > 0 else ""
        print(f"{m:>22} {d[0.05]:>9.4f} {d[0.10]:>9.4f} {delta:>+9.4f}{flag}")
print()
print(f"{worse} of {len(bym)} methods get worse from 5% to 10%.")
print("Epochs are fixed, so doubling the data doubles the gradient steps.")

                method        5%       10%     delta
             diversity    1.7996    1.8103   +0.0107  WORSE with more data
                   ifd    1.7801    1.7977   +0.0177  WORSE with more data
   learning_percentage    1.8093    1.8111   +0.0018  WORSE with more data
            perplexity    1.9093    1.8594   -0.0499
                random    1.7965    1.8025   +0.0061  WORSE with more data

4 of 5 methods get worse from 5% to 10%.
Epochs are fixed, so doubling the data doubles the gradient steps.


## 6. RQ1 -- do method rankings survive re-tuning the learning rate?

Needs at least two learning rates and three methods present in each. Reports what is
missing rather than failing if the sweep is incomplete.

In [7]:
from stats import ranking_stability

sweep = [r for r in ok if r["study"] == "fast_lr_sweep"]
tmp = defaultdict(lambda: defaultdict(list))
for r in sweep:
    c = r["config"]
    tmp[f"lr={c['learning_rate']:g}"][c["selection_method"]].append(
        r["metrics"]["held_out_loss"])
by_lr = {k: {m: float(np.mean(v)) for m, v in d.items()} for k, d in tmp.items()}

common = set.intersection(*(set(v) for v in by_lr.values())) if by_lr else set()
if len(by_lr) < 2 or len(common) < 3:
    print(f"INCOMPLETE: {len(by_lr)} learning rate(s), {len(common)} shared method(s).")
    print("RQ1 needs >= 2 learning rates and >= 3 methods present in each.")
else:
    order = sorted(by_lr, key=lambda s: float(s.split("=")[1]))
    res = ranking_stability({k: {m: v[m] for m in common} for k, v in by_lr.items()},
                            higher_is_better=False, order=order)
    for c in order:
        print(f"  {c:>12}: " + " > ".join(res["orderings"][c]))
    print()
    print(f"  mean Spearman {res['mean_spearman']:+.3f} | min {res['min_spearman']:+.3f}")
    print(f"  stable at the 0.95 threshold: {res['stable_at_0.95']}")

INCOMPLETE: 1 learning rate(s), 3 shared method(s).
RQ1 needs >= 2 learning rates and >= 3 methods present in each.


## 7. Paired bootstrap against the random baseline

Pairs on held-out **examples**, removing the example-difficulty variance shared by
both arms -- far more sensitive than comparing two means. Requires the per-example
loss vectors written by `evaluate.py`.

In [8]:
from stats import holm_bonferroni, paired_bootstrap


def per_example(run):
    p = ROOT / "results/runs" / run["run_id"] / "held_out_per_example_loss.npy"
    return np.load(p) if p.exists() else None


baseline = {(r["config"]["ratio"], r["config"]["seed"]): r
            for r in grid if r["config"]["selection_method"] == "random"}

pvals, lines = {}, []
for r in grid:
    c = r["config"]
    if c["selection_method"] == "random":
        continue
    b = baseline.get((c["ratio"], c["seed"]))
    a_loss = per_example(r)
    b_loss = per_example(b) if b else None
    if a_loss is None or b_loss is None:
        continue
    cmp = paired_bootstrap(a_loss, b_loss)
    tag = f"{c['selection_method']}@{c['ratio']:g}/s{c['seed']}"
    pvals[tag] = cmp.p_value
    lines.append(f"  {tag:>28}  " + cmp.describe(c["selection_method"], "random"))

if not lines:
    print("No per-example loss files found. They live in results/runs/<run_id>/ and")
    print("must be pushed from the training machine before this cell can run.")
else:
    print(NL.join(lines))
    corrected = holm_bonferroni(pvals)
    print()
    print("Holm-Bonferroni across the grid:")
    for tag, h in corrected.items():
        if h["reject"]:
            print(f"  SURVIVES correction: {tag} (p={h['p']:.4f} <= {h['threshold']:.4f})")
    if not any(h["reject"] for h in corrected.values()):
        print("  NOTHING survives correction -- report that as the headline result.")

No per-example loss files found. They live in results/runs/<run_id>/ and
must be pushed from the training machine before this cell can run.


## 8. Figures

Writes `results/figures/*.pdf` (for LaTeX) and `*.png` at 300 dpi.

In [9]:
print(subprocess.run([sys.executable, str(ROOT / "scripts/figures.py")],
                     capture_output=True, text=True, cwd=ROOT).stdout)

rendering figures from 23 runs:
  fig1 skipped: no study1 runs
  fig2 skipped: 0 LRs x 0 shared methods
  fig3: needs Studies 1+3+4 complete; see analyze.py for the same numbers



In [10]:
from IPython.display import Image, display

for f in sorted(glob.glob(str(ROOT / "results/figures/*.png"))):
    if "_demo" not in f:
        print(Path(f).name)
        display(Image(filename=f))

## 9. Full analysis report

`scripts/analyze.py` is the single source of truth for every number in the paper. It
prints `INSIDE NOISE` for differences whose confidence interval straddles zero, and
refuses to report the budget-aware rule's parameters when the fit is degenerate.

In [11]:
print(subprocess.run([sys.executable, str(ROOT / "scripts/analyze.py"), "--all"],
                     capture_output=True, text=True, cwd=ROOT).stdout)

23 grid runs, 0 LR-sweep runs

RQ1 -- do selection-method rankings survive re-tuning the learning rate?
  not enough data yet: 1 LRs x 5 shared methods

RQ2 -- which methods win once SELECTION cost is added to training cost?

                method  ratio           class    sel%  train FLOPs  total FLOPs    loss
                random   0.05            free    0.0%    6.617e+14    6.617e+14  1.8059
                random   0.05            free    0.0%    6.617e+14    6.617e+14  1.8059
                random   0.05            free    0.0%    6.650e+14    6.650e+14  1.7870
             diversity   0.05   training-free   11.0%    7.599e+14    8.537e+14  1.7997
             diversity   0.05   training-free   11.0%    7.599e+14    8.537e+14  1.7996
            perplexity   0.05   training-free   72.0%    2.765e+14    9.883e+14  1.8953
            perplexity   0.05   training-free   72.0%    2.765e+14    9.883e+14  1.9232
            perplexity   0.05   training-free   72.0%    2.765e+14    